# 3D Gaussian Splatting — Colab Training & Novel View Rendering

**What this notebook does:**
1. Mounts Google Drive to load your pre-prepared `data/gsplat_input/`
2. Installs gsplat + dependencies (CUDA build)
3. Downloads gsplat example scripts (`simple_trainer.py`, COLMAP dataset loader)
4. Trains a 3DGS model for 7 000 iterations
5. Renders novel views on three trajectories (ellipse, interpolated, spiral)
6. Saves checkpoints and renders back to Google Drive

**Before running:**
- Runtime → Change runtime type → **T4 GPU** (or A100 for faster training)
- Upload the **entire project folder** (or at minimum `data/gsplat_input/`) to Google Drive at:
  `My Drive/NVS/data/gsplat_input/`
- Adjust `DRIVE_ROOT` below if you chose a different location

**Expected runtime:** ~30–40 min on T4 for 7k steps with ~80 images

## 0. Mount Drive & configure paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
from pathlib import Path

# --- CONFIGURE THESE PATHS ---
DRIVE_ROOT    = Path('/content/drive/MyDrive/NVS')
GSPLAT_INPUT  = DRIVE_ROOT / 'data' / 'gsplat_input'    # from COLMAP step on Mac
RESULT_DIR    = DRIVE_ROOT / 'outputs'                   # checkpoints + renders saved here
# ------------------------------

MAX_STEPS     = 7000
SH_DEGREE     = 3
TEST_EVERY    = 8

# Local working directories (on Colab disk, faster I/O during training)
LOCAL_DATA    = Path('/content/gsplat_input')
LOCAL_RESULT  = Path('/content/outputs')

RESULT_DIR.mkdir(parents=True, exist_ok=True)

# Verify input data
assert GSPLAT_INPUT.exists(), f'gsplat_input not found at {GSPLAT_INPUT} — check Drive path'
sparse = GSPLAT_INPUT / 'sparse' / '0'
sparse_alt = GSPLAT_INPUT / 'sparse'
if not sparse.exists():
    sparse = sparse_alt
assert (sparse / 'cameras.bin').exists(), f'No cameras.bin found in {sparse}'

n_images = len(list((GSPLAT_INPUT / 'images').glob('*'))) if (GSPLAT_INPUT / 'images').exists() else 0
print(f'Input data verified: {n_images} images, sparse model at {sparse}')

## 1. Install dependencies

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.version.cuda}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE"}')
assert torch.cuda.is_available(), 'GPU required — set Runtime > Change runtime type > T4 GPU'

In [ ]:
%%capture
!pip install gsplat
!pip install pycolmap opencv-python numpy scipy tqdm torchmetrics[image] \
             imageio imageio-ffmpeg Pillow tensorboard matplotlib viser tyro PyYAML
!pip install fused-ssim

In [ ]:
import gsplat
print(f'gsplat: {gsplat.__version__}')

## 2. Download gsplat example scripts

gsplat's `simple_trainer.py` and COLMAP dataset loader are in the GitHub examples dir — not bundled in the pip wheel.

In [ ]:
import subprocess, sys, urllib.request

EXAMPLES_DIR = Path('/content/gsplat_examples')
EXAMPLES_DIR.mkdir(exist_ok=True)
(EXAMPLES_DIR / 'datasets').mkdir(exist_ok=True)

import gsplat as _gs
gsplat_version = _gs.__version__
BASE_URL = f'https://raw.githubusercontent.com/nerfstudio-project/gsplat/v{gsplat_version}/examples'

files_to_fetch = {
    'simple_trainer.py'    : f'{BASE_URL}/simple_trainer.py',
    'utils.py'             : f'{BASE_URL}/utils.py',
    'datasets/__init__.py' : None,
    'datasets/colmap.py'   : f'{BASE_URL}/datasets/colmap.py',
    'datasets/normalize.py': f'{BASE_URL}/datasets/normalize.py',
    'datasets/traj.py'     : f'{BASE_URL}/datasets/traj.py',
}

for rel_path, url in files_to_fetch.items():
    dst = EXAMPLES_DIR / rel_path
    if url is None:
        dst.write_text('')
        print(f'  Created empty {rel_path}')
    else:
        try:
            urllib.request.urlretrieve(url, dst)
            print(f'  Downloaded {rel_path}')
        except Exception as e:
            print(f'  WARN: could not fetch {rel_path}: {e}')

print(f'\ngsplat examples at {EXAMPLES_DIR}')

In [ ]:
# ── Patch 1: simple_trainer.py — make optional viewer deps non-fatal ──────────
trainer_path = EXAMPLES_DIR / 'simple_trainer.py'
src = trainer_path.read_text()

viewer_patches = [
    ('from gsplat_viewer import GsplatViewer, GsplatRenderTabState',
     'try:\n    from gsplat_viewer import GsplatViewer, GsplatRenderTabState\nexcept ImportError:\n    GsplatViewer = None; GsplatRenderTabState = None'),
    ('from nerfview import CameraState, RenderTabState, apply_float_colormap',
     'try:\n    from nerfview import CameraState, RenderTabState, apply_float_colormap\nexcept ImportError:\n    CameraState = None; RenderTabState = None; apply_float_colormap = None'),
]
for old, new in viewer_patches:
    if old in src:
        src = src.replace(old, new)
        print(f'  simple_trainer.py: patched viewer import')

if 'GsplatViewer is None' not in src and 'disable_viewer' in src:
    src = src.replace(
        'if cfg.disable_viewer:',
        'if GsplatViewer is None or cfg.disable_viewer:'
    )
    print('  simple_trainer.py: forced disable_viewer when GsplatViewer unavailable')

trainer_path.write_text(src)

# ── Patch 2: datasets/colmap.py — SceneManager shim for pycolmap >= 3.x ──────
# pycolmap removed SceneManager in v3.x; replace the bare import with a
# try/except that falls back to a Reconstruction-based compatibility wrapper.
SCENE_MANAGER_SHIM = '''\
try:
    from pycolmap import SceneManager
except ImportError:
    import pycolmap as _pycolmap

    class _ImageWrapper:
        def __init__(self, img):
            self._img = img
            pose = img.cam_from_world()
            self._R = pose.rotation.matrix()
            self._t = pose.translation
        def R(self): return self._R
        @property
        def tvec(self): return self._t
        @property
        def camera_id(self): return self._img.camera_id
        @property
        def name(self): return self._img.name

    class _CameraWrapper:
        def __init__(self, cam):
            self._cam = cam
            p = cam.params
            model_name = str(cam.model).split(".")[-1]
            self.camera_type = model_name
            self.width = cam.width
            self.height = cam.height
            if model_name == "SIMPLE_PINHOLE":
                self.fx = self.fy = p[0]; self.cx = p[1]; self.cy = p[2]
                self.k1 = self.k2 = self.p1 = self.p2 = 0.0
            elif model_name == "PINHOLE":
                self.fx = p[0]; self.fy = p[1]; self.cx = p[2]; self.cy = p[3]
                self.k1 = self.k2 = self.p1 = self.p2 = 0.0
            elif model_name == "SIMPLE_RADIAL":
                self.fx = self.fy = p[0]; self.cx = p[1]; self.cy = p[2]
                self.k1 = p[3]; self.k2 = self.p1 = self.p2 = 0.0
            elif model_name == "RADIAL":
                self.fx = self.fy = p[0]; self.cx = p[1]; self.cy = p[2]
                self.k1 = p[3]; self.k2 = p[4]; self.p1 = self.p2 = 0.0
            elif model_name == "OPENCV":
                self.fx = p[0]; self.fy = p[1]; self.cx = p[2]; self.cy = p[3]
                self.k1 = p[4]; self.k2 = p[5]; self.p1 = p[6]; self.p2 = p[7]
            else:
                self.fx = self.fy = p[0]
                self.cx = p[1] if len(p) > 1 else 0
                self.cy = p[2] if len(p) > 2 else 0
                self.k1 = self.k2 = self.p1 = self.p2 = 0.0

    class SceneManager:
        def __init__(self, colmap_dir):
            self._recon = _pycolmap.Reconstruction(colmap_dir)
        def load_cameras(self): pass
        def load_images(self): pass
        def load_points3D(self): pass
        @property
        def images(self):
            return {iid: _ImageWrapper(img) for iid, img in self._recon.images.items()}
        @property
        def cameras(self):
            return {cid: _CameraWrapper(cam) for cid, cam in self._recon.cameras.items()}
        @property
        def points3D(self):
            import numpy as _np
            return _np.array([p.xyz for p in self._recon.points3D.values()], dtype='float32')
        @property
        def point3D_errors(self):
            import numpy as _np
            return _np.array([p.error for p in self._recon.points3D.values()], dtype='float32')
        @property
        def point3D_colors(self):
            import numpy as _np
            return _np.array([p.color[:3] for p in self._recon.points3D.values()], dtype='uint8')
        @property
        def name_to_image_id(self):
            return {img.name: iid for iid, img in self._recon.images.items()}
        @property
        def point3D_id_to_images(self):
            return {pid: [(e.image_id, e.point2D_idx) for e in pt.track.elements]
                    for pid, pt in self._recon.points3D.items()}
        @property
        def point3D_id_to_point3D_idx(self):
            return {pid: idx for idx, pid in enumerate(self._recon.points3D.keys())}
'''

colmap_path = EXAMPLES_DIR / 'datasets' / 'colmap.py'
colmap_src  = colmap_path.read_text()

OLD_IMPORT = 'from pycolmap import SceneManager'
if OLD_IMPORT in colmap_src:
    colmap_src = colmap_src.replace(OLD_IMPORT, SCENE_MANAGER_SHIM, 1)
    colmap_path.write_text(colmap_src)
    print('  datasets/colmap.py: injected SceneManager compatibility shim')
elif 'SceneManager shim' in colmap_src or 'except ImportError' in colmap_src:
    print('  datasets/colmap.py: shim already present, skipping')
else:
    print('  datasets/colmap.py: WARNING — could not find SceneManager import to patch')

print('All patches applied.')

## 3. Copy input data to local Colab disk

Training from Drive directly is slow. We copy to local disk first.

In [ ]:
import shutil, time

if LOCAL_DATA.exists():
    shutil.rmtree(LOCAL_DATA)

print(f'Copying {GSPLAT_INPUT} → {LOCAL_DATA} ...')
t0 = time.time()
shutil.copytree(GSPLAT_INPUT, LOCAL_DATA)
print(f'Done in {time.time()-t0:.1f}s')

LOCAL_RESULT.mkdir(parents=True, exist_ok=True)

# COLMAP 4.x puts the sparse model at sparse/ instead of sparse/0/
# gsplat's dataset loader expects sparse/0/ — fix it here
sparse_0    = LOCAL_DATA / 'sparse' / '0'
sparse_root = LOCAL_DATA / 'sparse'
if not sparse_0.exists() and (sparse_root / 'cameras.bin').exists():
    sparse_0.mkdir(parents=True)
    for f in list(sparse_root.glob('*.bin')) + list(sparse_root.glob('*.txt')):
        shutil.copy2(f, sparse_0 / f.name)
    print('Mirrored sparse model → sparse/0/ for gsplat compatibility')

print('Input data ready.')

## 4. Probe simple_trainer.py CLI (shows accepted flags)

Run this cell first — it prints the trainer's help output so you can verify flag names match the installed gsplat version.

In [ ]:
import sys, os, subprocess
sys.path.insert(0, str(EXAMPLES_DIR))

env = os.environ.copy()
env['PYTHONPATH'] = str(EXAMPLES_DIR) + ':' + env.get('PYTHONPATH', '')

trainer_script = EXAMPLES_DIR / 'simple_trainer.py'

# Print help to validate CLI — this always exits 0 so we ignore returncode
result = subprocess.run(
    [sys.executable, str(trainer_script), '--help'],
    env=env, capture_output=True, text=True
)
help_text = result.stdout + result.stderr
# Show first 80 lines — enough to see all top-level flags
for line in help_text.splitlines()[:80]:
    print(line)

## 5. Train 3D Gaussian Splatting model

Training output is streamed line-by-line so you can monitor PSNR progression.

In [ ]:
import subprocess, sys, os

def run_streaming(cmd: list, env: dict) -> int:
    """Run a subprocess and stream stdout+stderr to notebook output."""
    proc = subprocess.Popen(
        cmd, env=env,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )
    for line in proc.stdout:
        print(line, end='', flush=True)
    proc.wait()
    return proc.returncode


env = os.environ.copy()
env['PYTHONPATH'] = str(EXAMPLES_DIR) + ':' + env.get('PYTHONPATH', '')

# tyro boolean fields must be passed as --flag True/False (not bare --flag)
# tyro list fields: pass values as separate tokens after the flag
train_cmd = [
    sys.executable, str(EXAMPLES_DIR / 'simple_trainer.py'),
    'default',
    '--data-dir',               str(LOCAL_DATA),
    '--result-dir',             str(LOCAL_RESULT),
    '--data-factor',            '1',
    '--max-steps',              str(MAX_STEPS),
    '--sh-degree',              str(SH_DEGREE),
    '--ssim-lambda',            '0.2',
    '--antialiased',            'True',    # tyro bool: requires explicit True/False
    '--test-every',             str(TEST_EVERY),
    '--eval-steps',             '3000', '5000', str(MAX_STEPS),  # space-sep list
    '--save-steps',             '3000', str(MAX_STEPS),
    '--disable-viewer',         'True',
    '--strategy.refine-start-iter', '500',
    '--strategy.refine-stop-iter',  str(min(5000, MAX_STEPS - 500)),
    '--strategy.refine-every',      '100',
    '--strategy.reset-every',       '3000',
    '--strategy.grow-grad2d',       '0.0002',
    '--strategy.grow-scale3d',      '0.01',
    '--strategy.prune-scale3d',     '0.1',
]

print('Command:', ' '.join(train_cmd[:7]), '...')
print(f'Full args: {train_cmd[7:]}')
print('\n--- Training output ---\n')

rc = run_streaming(train_cmd, env)

print('\n--- End of training output ---')
if rc != 0:
    print(f'\nERROR: training exited with code {rc}')
    print('\nDiagnostic — check trainer help for correct flag names:')
    diag = subprocess.run(
        [sys.executable, str(EXAMPLES_DIR / 'simple_trainer.py'), 'default', '--help'],
        env=env, capture_output=True, text=True
    )
    for line in (diag.stdout + diag.stderr).splitlines()[:60]:
        print(line)
else:
    print('\nTraining complete!')

## 6. Render novel views

Generates views along three camera trajectories:
- **ellipse**: smooth orbit around the scene
- **interpolated**: B-spline through training poses
- **spiral**: zooming spiral (good for forward-facing scenes)

In [ ]:
# Find the latest checkpoint
ckpt_dir = LOCAL_RESULT / 'ckpts'
ckpts = sorted(ckpt_dir.glob('ckpt_*.pt')) if ckpt_dir.exists() else []

if not ckpts:
    ckpts = sorted(LOCAL_RESULT.glob('**/*.pt'))

if not ckpts:
    raise RuntimeError('No checkpoint found. Training may have failed — check cell 5 output.')

checkpoint = ckpts[-1]
print(f'Using checkpoint: {checkpoint}')

In [ ]:
import torch
import numpy as np
from PIL import Image
from math import degrees, acos

sys.path.insert(0, str(EXAMPLES_DIR))
from datasets.colmap import Parser
from datasets.traj import (
    generate_ellipse_path_z,
    generate_interpolated_path,
    generate_spiral_path,
)
from gsplat import rasterization


def load_splats(ckpt_path: Path, device: str) -> dict:
    ckpt = torch.load(ckpt_path, map_location=device)
    splats = ckpt.get('splats', ckpt)
    return {k: v.to(device) for k, v in splats.items()}


def render_frame(splats: dict, c2w: np.ndarray, K: np.ndarray,
                 width: int, height: int, sh_degree: int, device: str) -> np.ndarray:
    c2w_t   = torch.from_numpy(c2w).float().to(device)
    viewmat = torch.linalg.inv(c2w_t).unsqueeze(0)  # (1,4,4) world-to-camera
    K_t     = torch.from_numpy(K).float().to(device).unsqueeze(0)  # (1,3,3)

    means     = splats['means']
    quats     = splats['quats']
    scales    = torch.exp(splats['scales'])           # stored in log-space
    opacities = torch.sigmoid(splats['opacities'])    # stored as logits

    if 'sh0' in splats:
        sh0 = splats['sh0']
        shN = splats.get('shN', torch.zeros(sh0.shape[0], 0, 3, device=device))
        colors = torch.cat([sh0, shN], dim=1)
    else:
        colors = splats['colors']

    render_colors, _, _ = rasterization(
        means=means, quats=quats, scales=scales,
        opacities=opacities, colors=colors,
        viewmats=viewmat, Ks=K_t,
        width=width, height=height,
        sh_degree=sh_degree,
    )
    img = render_colors[0].clamp(0, 1).detach().cpu().numpy()
    return (img * 255).astype(np.uint8)


print('Render helpers defined.')

In [ ]:
from tqdm.notebook import tqdm

DEVICE = 'cuda'
VIEWS_PER_TRAJ = 4

novel_dir = LOCAL_RESULT / 'novel_views'
novel_dir.mkdir(exist_ok=True)

parser = Parser(
    data_dir=str(LOCAL_DATA),
    factor=1,
    normalize=True,
    test_every=TEST_EVERY,
)

first_cam = list(parser.cameras.values())[0]
H, W = int(first_cam.height), int(first_cam.width)
K = np.array([
    [first_cam.fx,          0, first_cam.cx],
    [         0, first_cam.fy, first_cam.cy],
    [         0,          0,             1],
], dtype=np.float32)

training_c2ws = np.stack([parser.images[i].camtoworld for i in range(len(parser.images))])
print(f'Scene: {len(training_c2ws)} training poses, image {W}x{H}')

splats = load_splats(checkpoint, DEVICE)
print(f'Loaded {splats["means"].shape[0]:,} Gaussians')


def is_novel(c2w_cand: np.ndarray, train_c2ws: np.ndarray, min_deg: float = 15.0) -> bool:
    ax = c2w_cand[:3, 2]
    return all(
        degrees(acos(float(np.clip(ax @ c[:3, 2], -1, 1)))) >= min_deg
        for c in train_c2ws
    )


trajectories = {
    'ellipse'     : generate_ellipse_path_z(training_c2ws, n_frames=120),
    'interpolated': generate_interpolated_path(training_c2ws, n_interp=5),
    'spiral'      : generate_spiral_path(training_c2ws, n_frames=120),
}

total_rendered = 0
for traj_name, c2ws_traj in trajectories.items():
    traj_dir = novel_dir / traj_name
    traj_dir.mkdir(exist_ok=True)

    novel_idx = [i for i, c2w in enumerate(c2ws_traj) if is_novel(c2w, training_c2ws)]
    if len(novel_idx) < VIEWS_PER_TRAJ:
        print(f'  {traj_name}: only {len(novel_idx)} novel frames, using all')
        selected = novel_idx
    else:
        step = len(novel_idx) // VIEWS_PER_TRAJ
        selected = novel_idx[::step][:VIEWS_PER_TRAJ]

    print(f'\n{traj_name}: rendering {len(selected)} views...')
    for rank, idx in enumerate(tqdm(selected, desc=traj_name)):
        with torch.no_grad():
            frame = render_frame(splats, c2ws_traj[idx], K, W, H, SH_DEGREE, DEVICE)
        Image.fromarray(frame).save(traj_dir / f'novel_{rank:04d}_frame{idx:04d}.png')
        total_rendered += 1

print(f'\nTotal: {total_rendered} novel views rendered → {novel_dir}')

## 7. Copy results back to Drive

In [ ]:
import shutil, time

drive_result = DRIVE_ROOT / 'outputs'
if drive_result.exists():
    shutil.rmtree(drive_result)

print(f'Copying results to Drive...')
t0 = time.time()
shutil.copytree(LOCAL_RESULT, drive_result)
print(f'Done in {time.time()-t0:.1f}s')

pngs = list(drive_result.rglob('*.png'))
print(f'\n{len(pngs)} PNG files saved to Drive:')
for p in sorted(pngs)[:12]:
    print(f'  {p.relative_to(DRIVE_ROOT)}')
if len(pngs) > 12:
    print(f'  ... and {len(pngs)-12} more')

## 8. Preview renders

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

renders = sorted(LOCAL_RESULT.rglob('novel_*.png'))
if not renders:
    print('No renders found — check cell 6 output.')
else:
    samples = renders[:12]
    cols = 4
    rows = (len(samples) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(16, 4 * rows))
    axes_flat = axes.flatten() if hasattr(axes, 'flatten') else [axes]
    for ax, p in zip(axes_flat, samples):
        ax.imshow(Image.open(p))
        ax.set_title(f'{p.parent.name}/{p.name}', fontsize=7)
        ax.axis('off')
    for ax in axes_flat[len(samples):]:
        ax.axis('off')
    plt.tight_layout()
    plt.show()

## Done!

Results on Drive:
- Checkpoints: `NVS/outputs/ckpts/`
- Novel views: `NVS/outputs/novel_views/{ellipse,interpolated,spiral}/`
- TensorBoard logs: `NVS/outputs/`

Download to your Mac after Drive sync:
```bash
cp -r '/path/to/drive/NVS/outputs/' outputs/
tensorboard --logdir outputs/
```